# Clase 7 — Preparación de fuentes de conocimiento para el chatbot RAG

En las clases anteriores ya trabajamos con:

- modelo de IA;
- prompt;
- parámetros de generación;
- historial de conversación;
- PLN mínimo: saludo, tono e intención.

En esta clase damos el siguiente paso: **elegir, crear y evaluar fuentes de conocimiento** que después va a usar el chatbot con RAG.

> Importante: esta clase todavía **no construye el RAG completo** ni responde usando IA con documentos.  
> Hoy preparamos las fuentes. En las clases 8 y 9 vamos a leer, limpiar y fragmentar los documentos.

Para practicar vamos a usar un paquete de **5 fuentes demo ficticias**:

1. `01_reglamento_academico_2026.pdf`
2. `02_guia_inscripcion_ingreso_2026.pdf`
3. `03_horarios_contacto_servicios_2026.pdf`
4. `04_preguntas_frecuentes_chatbot_2026.pdf`
5. `05_fuente_no_oficial_para_descartar.pdf`

Las fuentes 1 a 4 sirven como base de conocimiento.  
La fuente 5 está incluida a propósito para aprender a detectar documentos dudosos o no oficiales.


## Qué cambiamos respecto del enfoque anterior

Antes usábamos mucha limpieza de texto porque el chatbot era más parecido a un chatbot por reglas.

Ahora usamos IA, entonces dejamos una lógica más realista:

```text
Pregunta original → se manda a la IA.
Texto normalizado → se usa solo para análisis interno.
```

En esta clase no vamos a corregir toda la pregunta. Solo vamos a usar PLN mínimo para detectar señales útiles y preparar mejor el proyecto.


## Objetivos de la clase

Al finalizar, cada grupo debería tener:

1. Una temática clara para su chatbot.
2. Un usuario objetivo definido.
3. Una configuración base del chatbot.
4. Un inventario inicial de fuentes.
5. Una evaluación de calidad de cada fuente.
6. Una decisión sobre qué fuentes usar, revisar o descartar.
7. Preguntas de prueba vinculadas a las fuentes.
8. Límites claros sobre qué debe y qué no debe responder.
9. Documentos propios preparados para trabajar en las próximas clases.

En esta clase vamos a trabajar con las **5 fuentes demo**, pero el objetivo final es que cada grupo pueda preparar sus propias fuentes.


In [ ]:
# Instalación condicional
# No instalamos pandas porque Colab normalmente ya lo trae.
# Solo instalamos lo que falte.

import importlib.util
import subprocess
import sys

paquetes = [
    ("pypdf", "pypdf"),
    ("docx", "python-docx"),
    ("gradio", "gradio")
]

faltan = []

for modulo, paquete in paquetes:
    if importlib.util.find_spec(modulo) is None:
        faltan.append(paquete)

if faltan:
    print("Faltan paquetes. Instalando:", faltan)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + faltan)
else:
    print("Todas las librerías necesarias ya están instaladas.")


In [ ]:
from pathlib import Path
from datetime import datetime
import zipfile
import re
import unicodedata

import pandas as pd
from pypdf import PdfReader
from docx import Document

print("Entorno listo")


# 1. Así configuré mi chatbot hasta ahora

Esta configuración todavía no carga el modelo en esta clase. Sirve para que cada grupo vaya dejando definido cómo quiere que se comporte su chatbot.

La idea es que no todos tengan el mismo chatbot: cada grupo debe adaptar nombre, temática, usuario, tono y límites.


In [ ]:
CONFIG_CHATBOT = {
    "nombre_chatbot": "AulaIA",
    "tematica": "Orientación académica institucional",
    "usuario_objetivo": "Estudiantes e ingresantes",
    "modelo_generativo": "Qwen/Qwen2.5-1.5B-Instruct",
    "temperature": 0.3,
    "top_p": 0.85,
    "top_k": 40,
    "max_new_tokens": 180,
    "max_mensajes_historial": 6,
    "pln_usado": "saludo, tono e intención",
    "modo_respuesta": "Base + conocimiento general controlado"
}

pd.DataFrame([CONFIG_CHATBOT])


## Para completar por grupo

Modifiquen los datos de abajo según su proyecto.


In [ ]:
proyecto = {
    "nombre_chatbot": "MiChatbot",
    "tematica": "Orientación académica",
    "usuario_objetivo": "Estudiantes e ingresantes",
    "problema_que_resuelve": "Responder consultas frecuentes usando fuentes confiables.",
    "tono_general": "claro, amable y educativo",
    "cuando_no_sabe": "indica que no tiene información suficiente en la base de conocimiento"
}

pd.DataFrame([proyecto])


# 2. PLN mínimo que conservamos

No usamos limpieza pesada ni diccionarios enormes de correcciones.

Conservamos solo lo necesario para analizar la consulta:

- normalizar para comparar;
- detectar saludo;
- detectar tono;
- detectar intención aproximada.

Ese análisis puede usarse después para armar mejor el prompt de la IA.


In [ ]:
def quitar_tildes(texto):
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(caracter for caracter in texto if unicodedata.category(caracter) != "Mn")
    return texto


def reducir_repeticiones(texto):
    # Reduce letras repetidas exageradas: holaaaa -> holaa
    return re.sub(r"(.)\1{2,}", r"\1\1", texto)


def normalizar_minimo(texto):
    texto = str(texto).lower().strip()
    texto = quitar_tildes(texto)
    texto = reducir_repeticiones(texto)
    texto = re.sub(r"[^a-z0-9ñü\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def detectar_saludo(texto_normalizado):
    saludos = ["hola", "buen dia", "buenas", "buenas tardes", "buenas noches", "holi"]
    return any(saludo in texto_normalizado for saludo in saludos)


def detectar_tono(texto_original, texto_normalizado):
    texto_original = str(texto_original)

    palabras_molesto = ["nadie responde", "hace rato", "urgente", "no entiendo nada", "malisimo", "reclamo"]
    palabras_formal = ["buen dia", "buenas tardes", "quisiera", "podria", "por favor", "disculpe"]
    palabras_informal = ["che", "porfa", "me decis", "dale", "holi"]
    palabras_agradecido = ["gracias", "genial", "perfecto", "muchas gracias"]

    if texto_original.isupper() and len(texto_original) > 8:
        return "molesto"
    if any(palabra in texto_normalizado for palabra in palabras_molesto):
        return "molesto"
    if any(palabra in texto_normalizado for palabra in palabras_agradecido):
        return "agradecido"
    if any(palabra in texto_normalizado for palabra in palabras_formal):
        return "formal"
    if any(palabra in texto_normalizado for palabra in palabras_informal):
        return "informal"

    return "neutro"


def detectar_intencion(texto_normalizado):
    intenciones = []

    if any(p in texto_normalizado for p in ["inscripcion", "inscribirme", "anotarme", "ingreso"]):
        intenciones.append("inscripción")
    if any(p in texto_normalizado for p in ["horario", "hora", "cuando atienden"]):
        intenciones.append("horarios")
    if any(p in texto_normalizado for p in ["requisito", "documentacion", "papeles", "dni"]):
        intenciones.append("requisitos")
    if any(p in texto_normalizado for p in ["telefono", "contacto", "mail", "correo", "direccion"]):
        intenciones.append("contacto")
    if any(p in texto_normalizado for p in ["regularidad", "final", "asistencia", "materia", "reglamento"]):
        intenciones.append("reglamento académico")
    if any(p in texto_normalizado for p in ["dato personal", "dni de otro", "telefono de otro", "alumno"]):
        intenciones.append("privacidad")

    if not intenciones:
        intenciones.append("general")

    return intenciones


def analizar_pln(texto):
    normalizado = normalizar_minimo(texto)
    return {
        "pregunta_original": texto,
        "texto_normalizado": normalizado,
        "saludo": detectar_saludo(normalizado),
        "tono": detectar_tono(texto, normalizado),
        "intenciones": detectar_intencion(normalizado)
    }

analizar_pln("Buenas tardes, quisiera saber los requisitos de inscripción por favor")


## Cómo usamos el tono detectado

El tono no sirve solo para mostrar una etiqueta. Lo usamos para generar una instrucción que después puede sumarse al prompt de la IA.


In [ ]:
def generar_instruccion_tono(tono):
    if tono == "formal":
        return "Respondé con tono formal, claro e institucional."
    if tono == "informal":
        return "Respondé con tono cercano, simple y natural, sin exagerar la informalidad."
    if tono == "molesto":
        return "Respondé con calma, empatía y claridad. No discutas con el usuario."
    if tono == "agradecido":
        return "Respondé de forma breve, amable y cordial."
    return "Respondé con tono claro, educativo y amable."


def armar_contexto_pln_para_prompt(pregunta):
    analisis = analizar_pln(pregunta)
    instruccion_tono = generar_instruccion_tono(analisis["tono"])

    contexto = f"""
Análisis PLN de la consulta:
- Tono detectado: {analisis['tono']}
- Intenciones detectadas: {', '.join(analisis['intenciones'])}
- Instrucción de estilo: {instruccion_tono}
""".strip()

    return contexto, analisis

contexto, analisis = armar_contexto_pln_para_prompt("che no entiendo nada, me decis los horarios?")
print(contexto)


# 3. Inventario inicial de fuentes

Cada fuente debe tener una función clara. No se cargan documentos “por las dudas”.

Para cada fuente pensá:

- qué información aporta;
- quién la produjo;
- si está actualizada;
- qué preguntas puede responder;
- si tiene texto extraíble;
- si contradice otra fuente;
- si sirve para RAG;
- si debe usarse, revisarse o descartarse.

En este ejemplo usamos 5 fuentes ficticias. Las primeras cuatro son útiles para el chatbot. La quinta es una fuente no oficial pensada para que el grupo aprenda a descartarla.


In [ ]:
fuentes = [
    {
        "fuente": "01_reglamento_academico_2026.pdf",
        "tipo": "PDF",
        "que_aporta": "regularidad, finales, asistencia, promoción, condiciones académicas",
        "origen": "Institución / Secretaría Académica",
        "anio": 2026,
        "estado": "vigente",
        "texto_extraible": "sí",
        "sirve_para_rag": "sí",
        "prioridad": "alta",
        "decision_inicial": "usar",
        "preguntas_que_responde": "¿cuánto dura la regularidad?, ¿cómo me inscribo a final?, ¿qué pasa con la asistencia?"
    },
    {
        "fuente": "02_guia_inscripcion_ingreso_2026.pdf",
        "tipo": "PDF",
        "que_aporta": "preinscripción, inscripción definitiva, documentación, fechas de ejemplo, pasos de ingreso",
        "origen": "Institución / Departamento de Alumnos",
        "anio": 2026,
        "estado": "vigente",
        "texto_extraible": "sí",
        "sirve_para_rag": "sí",
        "prioridad": "alta",
        "decision_inicial": "usar",
        "preguntas_que_responde": "¿qué necesito para inscribirme?, ¿qué documentación debo presentar?, ¿cómo hago el ingreso?"
    },
    {
        "fuente": "03_horarios_contacto_servicios_2026.pdf",
        "tipo": "PDF",
        "que_aporta": "horarios de atención, contacto institucional, áreas de consulta, servicios disponibles",
        "origen": "Institución / Secretaría",
        "anio": 2026,
        "estado": "vigente",
        "texto_extraible": "sí",
        "sirve_para_rag": "sí",
        "prioridad": "alta",
        "decision_inicial": "usar",
        "preguntas_que_responde": "¿cuándo atiende Secretaría?, ¿cuál es el contacto?, ¿dónde consulto por equivalencias?"
    },
    {
        "fuente": "04_preguntas_frecuentes_chatbot_2026.pdf",
        "tipo": "PDF",
        "que_aporta": "preguntas frecuentes redactadas en lenguaje simple para el chatbot",
        "origen": "Equipo del proyecto / revisado por la cátedra",
        "anio": 2026,
        "estado": "revisar",
        "texto_extraible": "sí",
        "sirve_para_rag": "sí",
        "prioridad": "media",
        "decision_inicial": "usar con revisión",
        "preguntas_que_responde": "¿qué puedo preguntarle al chatbot?, ¿qué temas responde?, ¿qué no puede responder?"
    },
    {
        "fuente": "05_fuente_no_oficial_para_descartar.pdf",
        "tipo": "PDF",
        "que_aporta": "información dudosa, incompleta o no confirmada",
        "origen": "No oficial / sin responsable claro",
        "anio": 2026,
        "estado": "dudoso",
        "texto_extraible": "sí",
        "sirve_para_rag": "no",
        "prioridad": "baja",
        "decision_inicial": "descartar",
        "preguntas_que_responde": "sirve para practicar evaluación de calidad, no para responder al usuario final"
    }
]

df_fuentes = pd.DataFrame(fuentes)
df_fuentes


# 4. Evaluación de calidad de las fuentes

Puntaje de 0 a 2:

| Puntaje | Significado |
|---|---|
| 0 | no cumple |
| 1 | cumple parcialmente |
| 2 | cumple bien |

Criterios:

- **confiabilidad:** sabemos de dónde viene;
- **relevancia:** sirve para la temática;
- **actualidad:** no está vieja;
- **claridad:** se entiende;
- **suficiencia:** cubre información importante;
- **coherencia:** no contradice otras fuentes.


In [ ]:
calidad_fuentes = [
    {
        "fuente": "01_reglamento_academico_2026.pdf",
        "confiabilidad": 2,
        "relevancia": 2,
        "actualidad": 2,
        "claridad": 1,
        "suficiencia": 2,
        "coherencia": 2,
        "observacion": "Fuente principal. Tiene información académica importante, aunque puede requerir explicación porque usa lenguaje formal."
    },
    {
        "fuente": "02_guia_inscripcion_ingreso_2026.pdf",
        "confiabilidad": 2,
        "relevancia": 2,
        "actualidad": 2,
        "claridad": 2,
        "suficiencia": 2,
        "coherencia": 2,
        "observacion": "Fuente central para preguntas de ingresantes e inscripción."
    },
    {
        "fuente": "03_horarios_contacto_servicios_2026.pdf",
        "confiabilidad": 2,
        "relevancia": 2,
        "actualidad": 2,
        "claridad": 2,
        "suficiencia": 1,
        "coherencia": 2,
        "observacion": "Fuente útil para contacto y horarios. Puede necesitar actualizarse si cambian los datos institucionales."
    },
    {
        "fuente": "04_preguntas_frecuentes_chatbot_2026.pdf",
        "confiabilidad": 1,
        "relevancia": 2,
        "actualidad": 2,
        "claridad": 2,
        "suficiencia": 1,
        "coherencia": 2,
        "observacion": "Muy clara para el chatbot, pero debe revisarse porque es una fuente elaborada para apoyo y no siempre reemplaza documentos oficiales."
    },
    {
        "fuente": "05_fuente_no_oficial_para_descartar.pdf",
        "confiabilidad": 0,
        "relevancia": 1,
        "actualidad": 0,
        "claridad": 1,
        "suficiencia": 0,
        "coherencia": 0,
        "observacion": "Fuente dudosa/no oficial. No debería usarse como base para responder datos institucionales."
    }
]

df_calidad = pd.DataFrame(calidad_fuentes)

criterios = [
    "confiabilidad", "relevancia", "actualidad",
    "claridad", "suficiencia", "coherencia"
]

df_calidad["puntaje_total"] = df_calidad[criterios].sum(axis=1)

def decidir_uso(puntaje):
    if puntaje >= 10:
        return "usar"
    if puntaje >= 7:
        return "revisar"
    return "no usar"

df_calidad["decision"] = df_calidad["puntaje_total"].apply(decidir_uso)
df_calidad


# 5. Revisión rápida de archivos reales

Esta parte comprueba si el archivo tiene texto extraíble.

Importante:

- Un PDF digital normalmente funciona.
- Un PDF escaneado puede devolver poco o nada de texto.
- Si no se puede extraer texto, más adelante se necesitaría OCR.
- En esta clase no hacemos OCR.


In [ ]:
def extraer_texto_pdf(ruta):
    lector = PdfReader(ruta)
    paginas = []

    for pagina in lector.pages:
        texto = pagina.extract_text() or ""
        if texto.strip():
            paginas.append(texto)

    return "
".join(paginas)


def extraer_texto_docx(ruta):
    documento = Document(ruta)
    parrafos = [
        parrafo.text
        for parrafo in documento.paragraphs
        if parrafo.text.strip()
    ]
    return "
".join(parrafos)


def leer_archivo(ruta):
    ruta = Path(ruta)
    extension = ruta.suffix.lower()

    if extension == ".pdf":
        return extraer_texto_pdf(ruta)

    if extension == ".docx":
        return extraer_texto_docx(ruta)

    if extension in [".txt", ".md"]:
        return ruta.read_text(encoding="utf-8", errors="ignore")

    raise ValueError(f"Formato no soportado: {extension}")


def sugerir_uso_por_nombre(nombre_archivo):
    nombre = normalizar_minimo(nombre_archivo)

    if "no oficial" in nombre or "descartar" in nombre:
        return "descartar"
    if "reglamento" in nombre:
        return "usar como fuente principal"
    if "inscripcion" in nombre or "ingreso" in nombre:
        return "usar como fuente principal"
    if "horarios" in nombre or "contacto" in nombre or "servicios" in nombre:
        return "usar como fuente de consulta frecuente"
    if "preguntas frecuentes" in nombre or "faq" in nombre or "chatbot" in nombre:
        return "usar como apoyo / revisar"

    return "revisar manualmente"


def resumen_archivo(ruta):
    texto = leer_archivo(ruta)
    texto_limpio = " ".join(texto.split())

    if len(texto_limpio) == 0:
        estado_lectura = "sin texto extraíble"
    elif len(texto_limpio) < 300:
        estado_lectura = "texto escaso / revisar"
    else:
        estado_lectura = "texto extraíble"

    return {
        "archivo": Path(ruta).name,
        "extension": Path(ruta).suffix.lower(),
        "caracteres_extraidos": len(texto_limpio),
        "estado_lectura": estado_lectura,
        "uso_sugerido": sugerir_uso_por_nombre(Path(ruta).name),
        "muestra": texto_limpio[:500]
    }


## Subir archivos para revisar

Ejecutá la siguiente celda y seleccioná tus documentos.

Para esta clase, subí el ZIP de fuentes demo o los 5 PDFs por separado:

- `01_reglamento_academico_2026.pdf`
- `02_guia_inscripcion_ingreso_2026.pdf`
- `03_horarios_contacto_servicios_2026.pdf`
- `04_preguntas_frecuentes_chatbot_2026.pdf`
- `05_fuente_no_oficial_para_descartar.pdf`

Formatos aceptados:

- `.zip`
- `.pdf`
- `.docx`
- `.txt`
- `.md`


In [ ]:
FORMATOS_VALIDOS = [".pdf", ".docx", ".txt", ".md"]


def preparar_archivos_subidos(archivos_subidos):
    rutas = []
    carpeta_extraida = Path("fuentes_extraidas")
    carpeta_extraida.mkdir(exist_ok=True)

    for nombre in archivos_subidos.keys():
        ruta = Path(nombre)
        extension = ruta.suffix.lower()

        if extension == ".zip":
            with zipfile.ZipFile(ruta, "r") as zip_ref:
                zip_ref.extractall(carpeta_extraida)

            for archivo in carpeta_extraida.rglob("*"):
                if archivo.is_file() and archivo.suffix.lower() in FORMATOS_VALIDOS:
                    rutas.append(str(archivo))

        elif extension in FORMATOS_VALIDOS:
            rutas.append(str(ruta))

        else:
            print("Archivo ignorado por formato no soportado:", ruta.name)

    return rutas


try:
    from google.colab import files
    archivos_subidos = files.upload()
    rutas_archivos = preparar_archivos_subidos(archivos_subidos)
except Exception:
    print("Esta celda está pensada para Google Colab. Si estás en VS Code, cargá las rutas manualmente.")
    rutas_archivos = []

print("Archivos detectados para revisar:")
for ruta in rutas_archivos:
    print("-", ruta)


In [ ]:
resumenes = []

for ruta in rutas_archivos:
    try:
        resumenes.append(resumen_archivo(ruta))
    except Exception as error:
        resumenes.append({
            "archivo": Path(ruta).name,
            "extension": Path(ruta).suffix.lower(),
            "caracteres_extraidos": 0,
            "estado_lectura": "error",
            "muestra": f"Error al leer archivo: {error}"
        })

df_revision_archivos = pd.DataFrame(resumenes)
df_revision_archivos


# 6. Preguntas que debería responder la base

Antes de construir el RAG, necesitamos preguntas de prueba.

No todas las preguntas son iguales:

- **documental:** debería responderse con una fuente.
- **general:** puede responderse con IA general en modo mixto, aclarando que no viene de la base.
- **no permitida:** no debería responderse.
- **dudosa:** aparece en una fuente no oficial, por eso no debería tomarse como válida.

Esto se relaciona con los modos que venimos trabajando:

```text
Solo base de conocimiento → responde solo si hay fuente confiable.
Base + conocimiento general → usa fuente si existe; si no existe y es general, puede responder aclarando.
Solo IA general → responde sin usar fuentes.
```


In [ ]:
preguntas_prueba = [
    {
        "pregunta": "¿Qué necesito para inscribirme?",
        "tipo_pregunta": "documental",
        "fuente_esperada": "02_guia_inscripcion_ingreso_2026.pdf",
        "deberia_responder_solo_base": "sí",
        "deberia_responder_base_ia": "sí"
    },
    {
        "pregunta": "¿Cuánto dura la regularidad de una materia?",
        "tipo_pregunta": "documental",
        "fuente_esperada": "01_reglamento_academico_2026.pdf",
        "deberia_responder_solo_base": "sí",
        "deberia_responder_base_ia": "sí"
    },
    {
        "pregunta": "¿Cuál es el horario de atención de Secretaría?",
        "tipo_pregunta": "documental",
        "fuente_esperada": "03_horarios_contacto_servicios_2026.pdf",
        "deberia_responder_solo_base": "sí",
        "deberia_responder_base_ia": "sí"
    },
    {
        "pregunta": "¿Qué temas puede responder el chatbot?",
        "tipo_pregunta": "documental",
        "fuente_esperada": "04_preguntas_frecuentes_chatbot_2026.pdf",
        "deberia_responder_solo_base": "sí",
        "deberia_responder_base_ia": "sí"
    },
    {
        "pregunta": "¿La cuota aumentó sin aviso?",
        "tipo_pregunta": "dudosa",
        "fuente_esperada": "05_fuente_no_oficial_para_descartar.pdf",
        "deberia_responder_solo_base": "no, porque la fuente es no oficial",
        "deberia_responder_base_ia": "no debería afirmar datos institucionales sin fuente confiable"
    },
    {
        "pregunta": "¿Por qué tengo frío?",
        "tipo_pregunta": "general",
        "fuente_esperada": "ninguna",
        "deberia_responder_solo_base": "no",
        "deberia_responder_base_ia": "sí, aclarando que es información general"
    },
    {
        "pregunta": "¿Me podés dar datos personales de otros alumnos?",
        "tipo_pregunta": "no permitida",
        "fuente_esperada": "ninguna",
        "deberia_responder_solo_base": "no",
        "deberia_responder_base_ia": "no"
    }
]

df_preguntas = pd.DataFrame(preguntas_prueba)
df_preguntas


# 7. Límites del chatbot

Un buen chatbot no solamente sabe qué responder. También sabe cuándo decir que no tiene información suficiente.

En datos institucionales, fechas, teléfonos, requisitos o reglamentos, no debería inventar.


In [ ]:
limites_chatbot = [
    "No responder información personal de estudiantes.",
    "No inventar fechas, teléfonos, requisitos, autoridades ni reglamentos.",
    "Si la pregunta requiere fuente y no hay fuente, debe indicarlo.",
    "En modo Base + IA puede responder preguntas generales, aclarando que no provienen de la base documental.",
    "No reemplazar decisiones administrativas, docentes, médicas o legales.",
    "Debe responder con el tono definido por el proyecto y adaptarse al tono detectado del usuario."
]

pd.DataFrame({"limites": limites_chatbot})


# 8. Interfaz básica de prueba

Esta interfaz **no es el chatbot final** y no carga todavía el modelo generativo.

Sirve para revisar:

- si los archivos tienen texto extraíble;
- cómo queda el análisis PLN mínimo de una consulta;
- si la pregunta parece documental, general, no permitida o dudosa;
- qué fuente debería usarse en el futuro;
- qué fuentes deberían usarse, revisarse o descartarse;
- qué instrucción de tono se mandaría al modelo.

La usamos para evitar trabajar todo desde tablas o consola.


In [ ]:
import gradio as gr


def clasificar_tipo_pregunta(intenciones, pregunta_normalizada):
    if "privacidad" in intenciones:
        return "no permitida"

    palabras_dudosas = ["cuota aumento", "aumento sin aviso", "rumor", "dicen que", "me dijeron que"]
    if any(palabra in pregunta_normalizada for palabra in palabras_dudosas):
        return "dudosa / requiere fuente confiable"

    intenciones_documentales = [
        "inscripción", "horarios", "requisitos", "contacto",
        "reglamento académico", "preguntas frecuentes"
    ]

    if any(intencion in intenciones_documentales for intencion in intenciones):
        return "documental"

    return "general"


def fuente_esperada_por_intencion(intenciones):
    if "privacidad" in intenciones:
        return "Ninguna. La consulta debe rechazarse por privacidad."
    if "reglamento académico" in intenciones:
        return "01_reglamento_academico_2026.pdf"
    if "inscripción" in intenciones or "requisitos" in intenciones:
        return "02_guia_inscripcion_ingreso_2026.pdf"
    if "horarios" in intenciones or "contacto" in intenciones:
        return "03_horarios_contacto_servicios_2026.pdf"
    if "preguntas frecuentes" in intenciones:
        return "04_preguntas_frecuentes_chatbot_2026.pdf"
    return "Ninguna fuente específica. Es una pregunta general."


def recomendacion_modo(tipo_pregunta):
    if tipo_pregunta == "documental":
        return (
            "Debe buscarse en una fuente confiable. En esta clase todavía no respondemos con RAG; "
            "solo verificamos qué documento debería permitir responder más adelante."
        )
    if tipo_pregunta == "general":
        return "En modo Base + IA podría responder, aclarando que no viene de la base documental."
    if tipo_pregunta == "no permitida":
        return "No debería responder datos personales. Debe explicar el límite de forma clara."
    if "dudosa" in tipo_pregunta:
        return "No debería afirmar el dato si solo aparece en una fuente no oficial o dudosa."
    return "Revisar manualmente."


def obtener_ruta_archivo(archivo):
    if archivo is None:
        return None

    if isinstance(archivo, str):
        return archivo

    if isinstance(archivo, dict):
        if "path" in archivo:
            return archivo["path"]
        if "name" in archivo:
            return archivo["name"]

    if hasattr(archivo, "path"):
        return archivo.path

    if hasattr(archivo, "name"):
        return archivo.name

    return None


def preparar_archivos_interfaz(archivos):
    rutas = []
    carpeta_extraida = Path("fuentes_extraidas_interfaz")
    carpeta_extraida.mkdir(exist_ok=True)

    for archivo in archivos or []:
        ruta = obtener_ruta_archivo(archivo)
        if ruta is None:
            continue

        ruta = Path(ruta)
        extension = ruta.suffix.lower()

        if extension == ".zip":
            try:
                with zipfile.ZipFile(ruta, "r") as zip_ref:
                    zip_ref.extractall(carpeta_extraida)

                for archivo_extraido in carpeta_extraida.rglob("*"):
                    if archivo_extraido.is_file() and archivo_extraido.suffix.lower() in FORMATOS_VALIDOS:
                        rutas.append(str(archivo_extraido))
            except Exception as error:
                print("No se pudo extraer el ZIP:", error)

        elif extension in FORMATOS_VALIDOS:
            rutas.append(str(ruta))

    return rutas


def revisar_en_interfaz(pregunta, archivos):
    pregunta = "" if pregunta is None else str(pregunta).strip()

    if pregunta == "":
        pregunta = "No se ingresó ninguna pregunta."

    contexto_pln, analisis = armar_contexto_pln_para_prompt(pregunta)
    tipo = clasificar_tipo_pregunta(analisis["intenciones"], analisis["texto_normalizado"])
    fuente_esperada = fuente_esperada_por_intencion(analisis["intenciones"])
    recomendacion = recomendacion_modo(tipo)

    salida_pln = f"""
Pregunta original:
{analisis['pregunta_original']}

Texto normalizado para análisis interno:
{analisis['texto_normalizado']}

Saludo detectado: {analisis['saludo']}
Tono detectado: {analisis['tono']}
Intenciones detectadas: {', '.join(analisis['intenciones'])}
Tipo de pregunta: {tipo}

Instrucción de tono para el futuro prompt:
{generar_instruccion_tono(analisis['tono'])}
""".strip()

    salida_fuente = f"""
Estado de la consulta:
Esta interfaz NO responde todavía con RAG.

Fuente esperada o decisión:
{fuente_esperada}

Recomendación:
{recomendacion}

Próximo paso:
En las clases 8 y 9 vamos a leer, limpiar y fragmentar los documentos. Después, con embeddings y RAG, el chatbot podrá recuperar fragmentos y responder usando la fuente.
""".strip()

    filas = []
    rutas = preparar_archivos_interfaz(archivos)

    for ruta in rutas:
        try:
            filas.append(resumen_archivo(ruta))
        except Exception as error:
            filas.append({
                "archivo": Path(str(ruta)).name,
                "extension": Path(str(ruta)).suffix.lower(),
                "caracteres_extraidos": 0,
                "estado_lectura": "error",
                "uso_sugerido": "revisar manualmente",
                "muestra": str(error)[:500]
            })

    if filas:
        tabla = pd.DataFrame(filas)
    else:
        tabla = pd.DataFrame(columns=[
            "archivo", "extension", "caracteres_extraidos", "estado_lectura", "uso_sugerido", "muestra"
        ])

    return salida_pln, salida_fuente, tabla


with gr.Blocks() as demo_fuentes:
    gr.Markdown("# Clase 7 — Preparación y evaluación de fuentes")
    gr.Markdown(
        "Subí el ZIP con las 5 fuentes demo o los 5 PDFs por separado. "
        "La interfaz revisa texto extraíble, analiza la pregunta y muestra qué fuente debería usarse más adelante. "
        "Todavía no responde con RAG."
    )

    pregunta_interfaz = gr.Textbox(
        label="Pregunta de prueba",
        placeholder="Ejemplo: ¿Cuánto dura la regularidad de una materia?",
        lines=2
    )

    archivos_interfaz = gr.File(
        label="Fuentes de conocimiento demo o fuentes del grupo",
        file_count="multiple",
        file_types=[".zip", ".pdf", ".docx", ".txt", ".md"],
        type="filepath"
    )

    boton_revisar = gr.Button("Revisar fuentes y pregunta", variant="primary")

    salida_pln = gr.Textbox(
        label="1. Análisis PLN de la pregunta",
        lines=12
    )

    salida_fuente = gr.Textbox(
        label="2. Fuente esperada y recomendación",
        lines=10
    )

    tabla_archivos = gr.Dataframe(
        label="3. Revisión de archivos subidos",
        wrap=True
    )

    gr.Markdown("### Fuentes demo esperadas")
    gr.Markdown(
        """
        - 01_reglamento_academico_2026.pdf
        - 02_guia_inscripcion_ingreso_2026.pdf
        - 03_horarios_contacto_servicios_2026.pdf
        - 04_preguntas_frecuentes_chatbot_2026.pdf
        - 05_fuente_no_oficial_para_descartar.pdf
        """
    )

    gr.Markdown("### Preguntas sugeridas")
    gr.Markdown(
        """
        - ¿Cuánto dura la regularidad de una materia?
        - Buenas tardes, quisiera saber los requisitos de inscripción.
        - Che, ¿me decís los horarios?
        - Nadie responde, necesito saber cuándo atiende Secretaría.
        - ¿Qué temas puede responder el chatbot?
        - ¿La cuota aumentó sin aviso?
        - ¿Por qué tengo frío?
        - ¿Me podés pasar datos personales de otros alumnos?
        """
    )

    boton_revisar.click(
        fn=revisar_en_interfaz,
        inputs=[pregunta_interfaz, archivos_interfaz],
        outputs=[salida_pln, salida_fuente, tabla_archivos]
    )

# Ejecutar la interfaz
# demo_fuentes.launch(debug=True)


## Ejecutar interfaz

Cuando quieras usar la interfaz, quitá el comentario de la última línea de la celda anterior o ejecutá esta celda:


In [ ]:
demo_fuentes.launch(debug=True)


# 9. Exportar entrega de la clase

Esta celda genera archivos CSV y un ZIP con:

- configuración del chatbot;
- proyecto;
- inventario de fuentes;
- evaluación de calidad;
- revisión de archivos;
- preguntas de prueba;
- límites del chatbot.


In [ ]:
carpeta_salida = Path("/content") if Path("/content").exists() else Path.cwd()
fecha = datetime.now().strftime("%Y%m%d_%H%M")

rutas_generadas = []

def guardar_csv(dataframe, nombre):
    ruta = carpeta_salida / f"{nombre}_{fecha}.csv"
    dataframe.to_csv(ruta, index=False, encoding="utf-8-sig")
    rutas_generadas.append(ruta)
    return ruta

ruta_config = guardar_csv(pd.DataFrame([CONFIG_CHATBOT]), "clase7_config_chatbot")
ruta_proyecto = guardar_csv(pd.DataFrame([proyecto]), "clase7_proyecto")
ruta_fuentes = guardar_csv(df_fuentes, "clase7_fuentes")
ruta_calidad = guardar_csv(df_calidad, "clase7_calidad_fuentes")
ruta_preguntas = guardar_csv(df_preguntas, "clase7_preguntas_prueba")
ruta_limites = guardar_csv(pd.DataFrame({"limites": limites_chatbot}), "clase7_limites_chatbot")

if "df_revision_archivos" in globals():
    ruta_revision = guardar_csv(df_revision_archivos, "clase7_revision_archivos")

ruta_zip = carpeta_salida / f"entrega_clase7_fuentes_conocimiento_{fecha}.zip"

with zipfile.ZipFile(ruta_zip, "w") as zipf:
    for ruta in rutas_generadas:
        zipf.write(ruta, arcname=ruta.name)

print("Archivos generados:")
for ruta in rutas_generadas:
    print("-", ruta)

print("\nZIP de entrega:", ruta_zip)


# Entrega de la clase

Al finalizar, cada grupo debe entregar o guardar:

- configuración inicial de su chatbot;
- temática y usuario objetivo;
- inventario de las 5 fuentes demo o de sus fuentes reales;
- evaluación de calidad;
- decisión sobre cada fuente: usar, revisar o descartar;
- revisión de archivos reales si los subieron;
- cinco preguntas de prueba como mínimo;
- límites del chatbot.

Para esta clase, recuerden:

- La interfaz no responde todavía con RAG.
- Las fuentes 1 a 4 pueden usarse como base de conocimiento.
- La fuente 5 debe analizarse críticamente y normalmente descartarse.
- El chatbot no debe inventar datos institucionales si no tiene una fuente confiable.
- En las próximas clases vamos a leer, limpiar y fragmentar los documentos para que después sí pueda recuperar fragmentos y responder.

En la próxima clase vamos a trabajar con la **lectura de archivos y extracción de texto**, preparando los documentos para dividirlos en fragmentos.
